In [ ]:
!pip install langgraph langchain langchain-openai gradio requests python-dotenv

In [47]:
from typing import Annotated, Optional, List
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
import os
import re
import requests
import gradio as gr

from dotenv import load_dotenv
from langchain.agents import Tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

In [26]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
serp_api_key = os.getenv("SERPER_API_KEY")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set")

if not serp_api_key:
    raise ValueError("SERPER_API_KEY is not set")

if not pushover_token or not pushover_user:
    raise ValueError("PUSHOVER_TOKEN and PUSHOVER_USER are required")

In [27]:
def input_guardrail(user_input: str) -> str:
    blocked = ["hack", "illegal", "exploit", "malware"]
    if any(word in user_input.lower() for word in blocked):
        raise ValueError("Unsafe input detected.")
    return user_input.strip()

def output_guardrail(response: str) -> str:
    response = (response or "").strip()
    if not response:
        return "No valid response generated."
    return response

In [28]:
def google_search(query: str) -> str:
    """Search for current information using SERP API and return short results."""
    url = "https://serpapi.com/search.json"
    params = {"q": query, "api_key": serp_api_key}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    organic = data.get("organic_results", [])[:5]
    if not organic:
        return "No search results found."
    lines = []
    for item in organic:
        title = item.get("title", "No title")
        link = item.get("link", "")
        snippet = item.get("snippet", "")
        lines.append(f"{title}\n{link}\n{snippet}")
    return "\n\n".join(lines)

def send_push(text: str) -> str:
    """Send a push notification to the user."""
    pushover_url = "https://api.pushover.net/1/messages.json"
    r = requests.post(
        pushover_url,
        data={"token": pushover_token, "user": pushover_user, "message": text},
        timeout=30,
    )
    r.raise_for_status()
    return "Push notification sent."

def calculator(expression: str) -> str:
    """Evaluate a simple math expression."""
    if not re.fullmatch(r"[0-9\s\+\-\*\/\(\)\.]+", expression):
        return "Invalid math expression"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception:
        return "Invalid math expression"

notes_store = []

def save_note(note: str) -> str:
    """Save a note to the assistant memory store."""
    notes_store.append(note)
    return "Note saved."

def get_notes(_: str = "") -> str:
    """Return saved notes."""
    return "\n".join(notes_store) if notes_store else "No notes found." 

In [29]:
search_tool = Tool(
    name="google_search",
    func=google_search,
    description="useful for current events, recent facts, and web lookup"
)

push_tool = Tool(
    name="send_push_notification",
    func=send_push,
    description="useful for when you want to send a push notification"
)

calculator_tool = Tool(
    name="calculator",
    func=calculator,
    description="useful for math calculations"
)

save_note_tool = Tool(
    name="save_note",
    func=save_note,
    description="useful for saving a short personal note"
)

get_notes_tool = Tool(
    name="get_notes",
    func=get_notes,
    description="useful for retrieving saved notes"
)

worker_tools = [search_tool, push_tool, calculator_tool, save_note_tool, get_notes_tool]

In [30]:
class PlanDecision(BaseModel):
    need_tools_first: bool = Field(description="Whether the worker will likely need tools for the task.")
    success_criteria: str = Field(description="What a good final answer must achieve.")
    clarification_needed: bool = Field(description="Whether the user must clarify before proceeding.")
    clarification_question: Optional[str] = Field(default=None, description="Question to ask if clarification is needed.")

class EvaluatorDecision(BaseModel):
    feedback: str = Field(description="Brief feedback on the worker result.")
    success_criteria_met: bool = Field(description="Whether the final response satisfies the request.")
    user_input_needed: bool = Field(description="Whether more user input is needed.")

In [31]:
class State(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    success_criteria: Optional[str]
    feedback_on_work: Optional[str]
    user_input_needed: bool
    success_criteria_met: bool
    guard_ok: bool

In [32]:
planner_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
worker_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
evaluator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

planner_llm_with_output = planner_llm.with_structured_output(PlanDecision)
worker_llm_with_tools = worker_llm.bind_tools(worker_tools)
evaluator_llm_with_output = evaluator_llm.with_structured_output(EvaluatorDecision)

In [33]:
def format_conversation(messages) -> str:
    parts = []
    for m in messages:
        role = m.__class__.__name__
        parts.append(f"{role}: {getattr(m, 'content', '')}")
    return "\n".join(parts)

In [34]:
def guardrail_node(state: State):
    text = ""
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage):
            text = str(m.content)
            break
    try:
        clean = input_guardrail(text)
        return {"guard_ok": True, "messages": [HumanMessage(content=clean)]}
    except ValueError as e:
        return {"guard_ok": False, "messages": [AIMessage(content=f"Input blocked: {e}")]}

In [35]:
def route_after_guard(state: State):
    return "planner" if state.get("guard_ok") else END

In [36]:
def planner(state: State):
    system = SystemMessage(content="""You are a planning assistant for a productivity sidekick.
Decide whether the worker likely needs tools, what success looks like, and whether clarification is needed.
Keep decisions practical and brief.""")
    result = planner_llm_with_output.invoke([system] + state["messages"])
    messages = [AIMessage(content=f"Planner success criteria: {result.success_criteria}")]
    if result.clarification_needed and result.clarification_question:
        messages.append(AIMessage(content=result.clarification_question))
    return {
        "messages": messages,
        "success_criteria": result.success_criteria,
        "user_input_needed": result.clarification_needed,
    }

In [37]:
def route_after_planner(state: State):
    if state.get("user_input_needed"):
        return END
    return "worker" 

In [38]:
def worker(state: State):
    criteria = state.get("success_criteria") or "Satisfy the user's request."
    system = SystemMessage(content=f"""You are a Productivity Sidekick.
Use tools when needed:
- calculator for math
- google_search for current or unknown info
- send_push_notification for reminders
- save_note and get_notes for note tasks

When you have enough information, write a final helpful answer with no tool call.
Success criteria: {criteria}
""")
    if state.get("feedback_on_work"):
        system = SystemMessage(content=system.content + f"\nPrevious feedback to address: {state['feedback_on_work']}")
    response = worker_llm_with_tools.invoke([system] + state["messages"])
    out = {"messages": [response]}
    return out

In [39]:
def worker_router(state: State):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "evaluator" 

In [40]:
worker_tool_node = ToolNode(worker_tools)

In [41]:
def evaluator(state: State):
    last = state["messages"][-1]
    last_response = getattr(last, "content", "") or str(last)

    system = SystemMessage(content="""You are an evaluator.
Judge whether the worker's latest answer satisfies the success criteria and whether user input is still needed.""")
    user = HumanMessage(content=f"""Conversation:
{format_conversation(state['messages'])}

Success criteria:
{state.get('success_criteria', '')}

Worker response to judge:
{last_response}
""")
    result = evaluator_llm_with_output.invoke([system, user])
    return {
        "messages": [AIMessage(content=f"Evaluator feedback: {result.feedback}")],
        "feedback_on_work": result.feedback,
        "success_criteria_met": result.success_criteria_met,
        "user_input_needed": result.user_input_needed,
    }

In [42]:
def route_after_evaluator(state: State):
    if state.get("success_criteria_met"):
        return "finalize"
    if state.get("user_input_needed"):
        return END
    return "worker" 

In [43]:
def finalize(state: State):
    for m in reversed(state["messages"]):
        if isinstance(m, AIMessage):
            text = getattr(m, "content", "")
            if text and not text.startswith("Evaluator feedback:") and not text.startswith("Planner success criteria:"):
                return {"messages": [AIMessage(content=output_guardrail(text))]}
    return {"messages": [AIMessage(content="No valid response generated.")]}

In [ ]:
memory = MemorySaver()

graph_builder = StateGraph(State)
graph_builder.add_node("guardrail", guardrail_node)
graph_builder.add_node("planner", planner)
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", worker_tool_node)
graph_builder.add_node("evaluator", evaluator)
graph_builder.add_node("finalize", finalize)

graph_builder.add_edge(START, "guardrail")
graph_builder.add_conditional_edges(
    "guardrail",
    route_after_guard,
    {END: END, "planner": "planner"}
)
graph_builder.add_conditional_edges(
    "planner",
    route_after_planner,
    {END: END, "worker": "worker"}
)
graph_builder.add_conditional_edges(
    "worker",
    worker_router,
    {"tools": "tools", "evaluator": "evaluator"}
)
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges(
    "evaluator",
    route_after_evaluator,
    {END: END, "worker": "worker", "finalize": "finalize"}
)
graph_builder.add_edge("finalize", END)

graph = graph_builder.compile(checkpointer=memory)

display(Image(graph.get_graph().draw_mermaid_png()))

In [45]:
def chat(user_input, history):
    try:
        result = graph.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config={"configurable": {"thread_id": "session-1"}}
        )
        final_text = ""
        for m in reversed(result["messages"]):
            if isinstance(m, AIMessage):
                text = getattr(m, "content", "")
                if text and not text.startswith("Evaluator feedback:") and not text.startswith("Planner success criteria:"):
                    final_text = text
                    break
        return output_guardrail(final_text)
    except Exception as e:
        return f"Error: {str(e)}" 

In [49]:
def stream_chat(user_input, history):
    response = chat(user_input, history)
    partial = ""
    for ch in response:
        partial += ch
        yield partial

In [ ]:
demo = gr.ChatInterface(
    fn=stream_chat,
    title=" Multi-Agent Productivity Sidekick",
    description="Planner + Worker + Tools + Evaluator with structured output"
)

demo.launch()